En négligeant le poids en octet du mot, déterminer le nombre d’octets moyen pour stocker la représentation vectorielle en 300 dimensions d’un seul mot :

ici, les valeurs stockées sont en floattants, donc pour connaitre le nombre d'octets moyen afin de stocker cette réprésentation en 300 dimension il faut prendre en considération deux cas :
1. le flottant est un float32 :
    si le flottant est un float32 alors celui-ci tient sur 4 octets, on peut donc en conclure que 4x300 = 1200 octets, ce qui représente le nombre d'octets moyen pour ce stockage.
2. le flottant est un float64 :
    si le flottant est un float64 alors celui-ci tient sur 8 octets, on peut donc en conclure que 8x300 = 2400 octets, ce qui représente le nombre d'octets moeyn pour ce stockage.

In [6]:
#coding: utf-8
import pandas as pd

File_Path_ = "data/vector-fr.txt"
Word_Search_ = "faire"
Sentence_ = "Je suis tombé du toit de l’usine"

Télécharger le jeu de données et créer une fonction avec les conditions suivantes :
- Prend en paramètre un mot à chercher dans le jeu de données
- Parcoure le jeu de données et retourne la représentation vectorielle de ce mot, ou « None » le cas
échéant.
- Ne doit jamais prendre en mémoire plus que 100Mo pour la gestion des données du jeu de données.
Pour ce faire, vous pouvez prendre votre calcul défini en 1. et retrouver le nombre de ligne « maximal » à
récupérer à la fois.

In [7]:
def Load_Vectors(_file_path, _block_size):
    with open(_file_path, 'r', encoding='utf-8') as f:
        while True:
            lines = []
            for i in range(_block_size):
                line = f.readline()
                if not line:
                    break
                lines.append(line.strip().split())
            if not lines:
                break
            yield lines

def Find_Word_Vector(_file_path, _word, _max_memory_mb=100):
    vector_size_bytes = 1200
    max_memory_bytes = _max_memory_mb * 1024 * 1024
    block_size = max_memory_bytes // vector_size_bytes
    
    for lines in Load_Vectors(_file_path, block_size):
        df = pd.DataFrame(lines)
        df.columns = ['word'] + [f'vec_{i}' for i in range(300)]
        
        word_row = df[df['word'] == _word]
        if not word_row.empty:
            vector = word_row.iloc[0, 1:].astype(float).tolist()
            return vector
    return None

Vector_ = Find_Word_Vector(File_Path_, Word_Search_)
print("Vector for '{}': {}".format(Word_Search_, Vector_))

Vector for 'faire': [0.0476, -0.0587, -0.0058, 0.0405, -0.1526, -0.0154, -0.0415, -0.0278, 0.0304, -0.0621, 0.0001, -0.0342, 0.009, 0.0328, 0.0813, 0.0445, -0.0278, 0.045, -0.0108, 0.034, -0.0181, 0.0078, -0.0197, -0.022, 0.032, 0.0264, 0.011, -0.0089, 0.1163, 0.036, -0.0052, 0.012, 0.0017, 0.0398, 0.0179, -0.0103, -0.1175, 0.0385, 0.0251, -0.018, -0.0322, 0.0006, -0.0095, -0.0167, -0.1183, -0.0093, -0.0104, 0.0025, 0.023, -0.0286, 0.0204, -0.0063, 0.0072, -0.0034, 0.0195, 0.118, -0.0647, 0.0734, 0.0985, 0.095, -0.0183, 0.0947, -0.0403, -0.0477, 0.0213, 0.0315, -0.0289, -0.0337, 0.0409, 0.0041, 0.0217, 0.027, 0.083, 0.0053, 0.0193, 0.0056, 0.0185, 0.0079, -0.0806, 0.0768, 0.0012, -0.0229, -0.117, -0.2044, -0.0594, -0.0049, 0.0062, -0.0037, -0.0293, 0.0178, 0.0104, 0.0307, -0.0115, -0.003, 0.0458, 0.0238, 0.0369, -0.1318, -0.016, 0.0602, 0.0373, 0.019, -0.0074, 0.0338, 0.0094, -0.001, -0.0023, 0.0154, -0.0144, 0.0669, 0.0688, 0.0563, 0.0242, -0.0272, 0.0488, -0.0798, -0.0293, 0.0211, -0

Faire un programme prenant dans la toute première cellule du notebook une phrase, par exemple « Je suis tombé du toit de l’usine ».

In [8]:
import re

def Sentence_Vector(_sentence, _file_path):
    words = re.findall(r'\w+|[^\w\s]', _sentence, re.UNICODE)
    word_set = set(words)
    word_vectors = {word: None for word in word_set}
    
    vector_size_bytes = 1200
    max_memory_bytes = 100 * 1024 * 1024
    block_size = max_memory_bytes // vector_size_bytes
    
    for lines in Load_Vectors(_file_path, block_size):
        df = pd.DataFrame(lines)
        df.columns = ['word'] + [f'vec_{i}' for i in range(300)]
        
        words_to_remove = []
        for word in word_set.copy():
            if word_vectors[word] is None:
                word_row = df[df['word'] == word]
                if not word_row.empty:
                    word_vectors[word] = word_row.iloc[0, 1:].astype(float).tolist()
                    words_to_remove.append(word)
        
        for word in words_to_remove:
            word_set.remove(word)
        
        if all(word_vectors[word] is not None for word in word_vectors):
            break
    
    valid_vectors = [vector for vector in word_vectors.values() if vector is not None]
    
    if not valid_vectors:
        return None
    
    sentence_vector = [sum(x) / len(x) for x in zip(*valid_vectors)]
    
    return sentence_vector

Sentence_Vector_ = Sentence_Vector(Sentence_, File_Path_)
print("Vector for the sentence '{}': {}".format(Sentence_, Sentence_Vector_))

Vector for the sentence 'Je suis tombé du toit de l’usine': [-0.02694444444444444, 0.06426666666666667, 0.035800000000000005, -0.033966666666666666, -0.09157777777777779, 0.005799999999999999, 0.05666666666666667, -0.03885555555555556, 0.002322222222222224, -0.1129, -0.032744444444444444, 0.04657777777777779, -0.01043333333333334, 0.0760888888888889, -0.047244444444444436, -0.0048000000000000004, -0.03484444444444444, 0.04544444444444444, -0.07478888888888889, 0.0594888888888889, -0.019933333333333334, -0.03636666666666667, -0.05743333333333332, 0.04762222222222223, 0.023999999999999997, 0.08056666666666668, -0.00853333333333333, 0.011944444444444445, -0.05136666666666667, 0.013411111111111108, -0.0114, -0.02971111111111111, -0.00797777777777778, -0.010666666666666666, 0.07617777777777779, 0.03543333333333333, -0.043122222222222215, 0.06697777777777776, -0.014666666666666668, 0.011577777777777782, -0.024911111111111113, -0.0021777777777777754, 0.04148888888888889, -0.00683333333333334,

Implémentez une fonction pour calculer la distance cosinus entre deux éléments avec l’équation suivante :
(𝐴∙𝐵) / (‖𝐴‖∙‖𝐵‖)sachant que vous pouvez utiliser les fonctions de Numpy DOT et NORM.

In [9]:
import numpy as np

def Cosine_Distance(_path, _word):
    dot_product = np.dot(_path, _word)
    norm_path = np.linalg.norm(_path)
    norm_word = np.linalg.norm(_word)
    return dot_product / (norm_path * norm_word)

Word_A = "faire"
Word_B = "deux"

Vector_A = Find_Word_Vector(File_Path_, Word_A)
Vector_B = Find_Word_Vector(File_Path_, Word_B) 

Distance_ = Cosine_Distance(Vector_A, Vector_B)
print("Cosine distance between '{}' and '{}': {}".format(Word_A, Word_B, Distance_))

KeyboardInterrupt: 

Utilisez la fonction de distance développée pour comparer chaque mot du jeu de données avec celui-ci.
Retourner le « mot » dont la représentation vectorielle est la plus proche de celle retrouvée en 3.

In [ ]:
def Find_Closest_Word(_sentence_vector, _file_path, _max_memory_mb=100):
    closest_word = None
    highest_similarity = -1  # Cosine similarity ranges from -1 to 1
    
    vector_size_bytes = 1200
    max_memory_bytes = _max_memory_mb * 1024 * 1024
    block_size = max_memory_bytes // vector_size_bytes
    
    for lines in Load_Vectors(_file_path, block_size):
        df = pd.DataFrame(lines)
        df.columns = ['word'] + [f'vec_{i}' for i in range(300)]
        
        vectors = df.iloc[:, 1:].astype(float).to_numpy()
        words = df['word'].to_numpy()
        
        similarities = np.dot(vectors, _sentence_vector) / (np.linalg.norm(vectors, axis=1) * np.linalg.norm(_sentence_vector))
        
        max_similarity_index = np.argmax(similarities)
        if similarities[max_similarity_index] > highest_similarity:
            highest_similarity = similarities[max_similarity_index]
            closest_word = words[max_similarity_index]
    
    return closest_word

if Sentence_Vector_ is not None:
    Closest_Word_ = Find_Closest_Word(Sentence_Vector_, File_Path_)
    print("Closest word to the sentence '{}': {}".format(Sentence_, Closest_Word_))
else:
    print("No valid vectors found for the sentence.")

KeyboardInterrupt: 